### Import Libraries

In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import cv2
import torch
import mlflow
import logging
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch import optim
from src.org_unet_parts import *
from dataloader import ACDCDataset
from utils.dice_score import dice_loss
from utils.evaluate import evaluate
%matplotlib inline

In [ ]:
log_file_name = f'./logs/acdc_feedback_unet_{datetime.now().strftime("%Y%m%d%H%M%S")}.log'
logging.basicConfig(filename=log_file_name, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Using device {device}')
print(f'Using device {device}')

In [ ]:
mlflow.login() # host: https://community.cloud.databricks.com/
mlflow.set_tracking_uri("databricks")

# Initialize logging - create a new MLflow Experiment
mlflow.set_experiment("/cmri-segmentation-of-ventricular-structures-and-myocardium")

In [ ]:
dir_checkpoint = Path('./models/checkpoints/feedback_unet_org/')

epochs = 5
batch_size = 8
lr = 1e-5
scale = 1
amp = False

### Data Loading

In [ ]:
root_dir = r'../data/ACDC/img_slices_org/'
logging.info(f'Using root_dir {root_dir}')

training_dataset = ACDCDataset(root_dir=root_dir, dataset='training', sequence=True)
validation_dataset = ACDCDataset(root_dir=root_dir, dataset='validation', sequence=True)
testing_dataset = ACDCDataset(root_dir=root_dir, dataset='testing', sequence=True)

logging.info(f'Training dataset size: {len(training_dataset)}')
logging.info(f'Validation dataset size: {len(validation_dataset)}')
logging.info(f'Testing dataset size: {len(testing_dataset)}')

# Load data from the dataset
train_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Display image and mask.
sample_batch = next(iter(test_dataloader))
img_idx = 0

print(f'Keys: {sample_batch.keys()}')
print(f"Image batch shape: {sample_batch['image'].size()}")
print(f"Mask batch shape: {sample_batch['masks'][0].size()}")

img = sample_batch['image'][img_idx].squeeze().numpy()
fig = plt.figure(figsize=(3, 3))
plt.imshow(img, cmap='gray')
plt.tight_layout()
plt.show()

In [ ]:
# Iterate through the dataset and plot the first 4 samples
n_samples_to_plot = 2

for i, sample in enumerate(test_dataloader):
    if i >= n_samples_to_plot:
        break
    
    # Extract data from the sample
    image = sample['image'][i].squeeze().numpy()  # Convert to numpy array
    msk_lv = sample['masks'][i][0].squeeze().numpy()
    msk_rv = sample['masks'][i][1].squeeze().numpy()
    msk_myo = sample['masks'][i][2].squeeze().numpy()
    
    # Create a figure with subplots
    fig, axes = plt.subplots(1, 4, figsize=(8, 5))
    
    # Plot the image and masks
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Image')
    axes[0].axis('off')
    
    axes[1].imshow(msk_lv, cmap='gray')
    axes[1].set_title('Mask LV')
    axes[1].axis('off')
    
    axes[2].imshow(msk_rv, cmap='gray')
    axes[2].set_title('Mask RV')
    axes[2].axis('off')
    
    axes[3].imshow(msk_myo, cmap='gray')
    axes[3].set_title('Mask Myo')
    axes[3].axis('off')
    
    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()

### Feedback U-Net Model

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size):
        super(ConvLSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2

        self.Wxi = nn.Conv2d(input_size, hidden_size, kernel_size, padding=self.padding)
        self.Whi = nn.Conv2d(hidden_size, hidden_size, kernel_size, padding=self.padding)
        self.Wxf = nn.Conv2d(input_size, hidden_size, kernel_size, padding=self.padding)
        self.Whf = nn.Conv2d(hidden_size, hidden_size, kernel_size, padding=self.padding)
        self.Wxc = nn.Conv2d(input_size, hidden_size, kernel_size, padding=self.padding)
        self.Whc = nn.Conv2d(hidden_size, hidden_size, kernel_size, padding=self.padding)
        self.Wxo = nn.Conv2d(input_size, hidden_size, kernel_size, padding=self.padding)
        self.Who = nn.Conv2d(hidden_size, hidden_size, kernel_size, padding=self.padding)

    def forward(self, x, h):
        ci, hi = h
        xi = self.Wxi(x) + self.Whi(hi)
        fi = torch.sigmoid(xi)
        xf = self.Wxf(x) + self.Whf(hi)
        ff = torch.sigmoid(xf)
        xc = self.Wxc(x) + self.Whc(hi)
        cc = fi * ci + ff * torch.tanh(xc)
        xo = self.Wxo(x) + self.Who(hi)
        ho = torch.tanh(xo) * torch.sigmoid(cc)
        return ho, (cc, ho)

class ConvLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size, num_layers):
        super(ConvLSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.kernel_size = kernel_size
        self.num_layers = num_layers
        self.seq_len = 1

        self.convlstm_cells = nn.ModuleList()
        for i in range(num_layers):
            input_dim = input_size if i == 0 else hidden_size
            self.convlstm_cells.append(ConvLSTMCell(input_dim, hidden_size, kernel_size))

    def forward(self, x):
        batch_size, _, height, width = x.size()
        hidden_state = self.init_hidden(batch_size, height, width)

        outputs = []
        for t in range(self.seq_len):
            input_tensor = x[:, t, :, :]
            for i in range(self.num_layers):
                # self.convlstm_cells[i].flatten_parameters()
                input_tensor, hidden_state[i] = self.convlstm_cells[i](input_tensor, hidden_state[i])
            outputs.append(input_tensor)

        outputs = torch.stack(outputs, dim=1)
        return outputs

    def init_hidden(self, batch_size, height, width):
        hidden_state = []
        for _ in range(self.num_layers):
            hidden_state.append((torch.zeros(batch_size, self.hidden_size, height, width).to(device),
                                 torch.zeros(batch_size, self.hidden_size, height, width).to(device)))
        return hidden_state

In [ ]:
class FeedbackUNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(FeedbackUNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Encoder layers
        self.enc1 = nn.Sequential(
            ConvLSTM(n_channels, 8, kernel_size=3, num_layers=2),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.enc2 = nn.Sequential(
            ConvLSTM(8, 16, kernel_size=3, num_layers=2),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.enc4 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.enc5 = nn.Sequential(
            ConvLSTM(64, 128, kernel_size=3, num_layers=2),
            nn.Dropout(0.5)
        )

        # Decoder layers
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            ConvLSTM(16, 16, kernel_size=3, num_layers=2)
        )

        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2),
            nn.Conv2d(16, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            ConvLSTM(8, 8, kernel_size=3, num_layers=2),
            nn.Conv2d(8, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True)
        )

        self.out = nn.Conv2d(8, n_classes, kernel_size=1)

    def forward(self, x):
        # First round
        # x = x.unsqueeze(1)
        # logging.info(f"tensor shape after unsqueeze: {x.shape}")
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        x5 = self.enc5(x4)

        x = self.dec1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.dec2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.dec3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.dec4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.out(x)

        # Second round
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        x5 = self.enc5(x4)

        x = self.dec1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.dec2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.dec3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.dec4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.out(x)

        return x

#### Feedback UNet - V2

In [ ]:
class ConvLSTM2D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super(ConvLSTM2D, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        return self.relu(x)

class FeedbackUNetV2(nn.Module):
    def __init__(self, in_channels, classes):
        super(FeedbackUNetV2, self).__init__()
        self.n_channels = in_channels
        self.n_classes = classes
        self.bilinear = False

        self.convlstm_a1 = ConvLSTM2D(in_channels, 8, kernel_size=3, padding=1)
        self.convlstm_a2 = ConvLSTM2D(8, 8, kernel_size=3, padding=1)
        self.convlstm_b1 = ConvLSTM2D(8, 16, kernel_size=3, padding=1)
        self.convlstm_b2 = ConvLSTM2D(16, 16, kernel_size=3, padding=1)
        self.conv_down1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv_down2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.convlstm_c1 = ConvLSTM2D(64, 128, kernel_size=3, padding=1)
        self.convlstm_c2 = ConvLSTM2D(128, 128, kernel_size=3, padding=1)
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.deconv3 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.conv_final = nn.Conv2d(16, classes, kernel_size=1)

    def forward(self, x):
        x = self.convlstm_a1(x)
        x = self.convlstm_a2(x)
        x = F.max_pool2d(x, 2)
        x = self.convlstm_b1(x)
        x = self.convlstm_b2(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn1(self.conv_down1(x)))
        x = F.max_pool2d(F.relu(self.bn2(self.conv_down2(x))), 2)
        x = self.convlstm_c1(x)
        x = self.convlstm_c2(x)
        x = self.deconv1(x)
        x = self.deconv2(x)
        x = self.deconv3(x)
        x = self.conv_final(x)
        return F.softmax(x, dim=1)

#### Feedback UNet - V3

In [ ]:
# Encoder block
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x_pooled = self.pool(x)
        return x, x_pooled

# Decoder block
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat((x, skip), dim=1)
        x = self.conv(x)
        return x

# ConvLSTM block
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias=True):
        super(ConvLSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.bias = bias

        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state

        combined = torch.cat([input_tensor, h_cur], dim=1)  # concatenate along channel axis
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(self, batch_size, image_size):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device))

# Feedback U-Net model
class FeedbackUNetV3(nn.Module):
    def __init__(self, n_channels=1, n_classes=3, bilinear=False):
        super(FeedbackUNetV3, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        
        self.enc1 = EncoderBlock(self.n_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.bottleneck1 = ConvLSTMCell(512, 512, kernel_size=3)
        self.bottleneck2 = ConvLSTMCell(512, 512, kernel_size=3)

        self.dec4 = DecoderBlock(512 + 512, 256)
        self.dec3 = DecoderBlock(256 + 256, 128)
        self.dec2 = DecoderBlock(128 + 128, 64)
        self.dec1 = DecoderBlock(64 + 64, 64)

        self.final_conv = nn.Conv2d(64, self.n_classes, kernel_size=1)

    def forward(self, x):
        batch_size, _, height, width = x.size()
        
        # First pass through the network
        # Encoder
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # Bottleneck with feedback (ConvLSTM)
        h, c = self.bottleneck1.init_hidden(batch_size, (height // 16, width // 16))
        x, (h, c) = self.bottleneck1(x, (h, c))

        # Decoder
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final Convolution
        output1 = self.final_conv(x)

        # Feedback loop - second round
        x = torch.cat((output1, x), dim=1)  # Feedback by concatenating the output with features

        # Second pass through bottleneck
        x, (h, c) = self.bottleneck2(x, (h, c))

        # Decoder
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final Convolution
        output2 = self.final_conv(x)
        return output2

### Model Training

In [ ]:
def train_model(
        model,
        device,
        epochs: int = 5,
        batch_size: int = 1,
        learning_rate: float = 1e-5,
        save_checkpoint: bool = True,
        img_scale: float = 0.5,
        amp: bool = False,
        weight_decay: float = 1e-8,
        momentum: float = 0.999,
        gradient_clipping: float = 1.0,
):
    n_train = len(training_dataset)
    n_val = len(validation_dataset)
    
    # Start an MLflow run
    with mlflow.start_run():
        logging.info(f'''Starting training:
            Epochs:          {epochs}
            Batch size:      {batch_size}
            Learning rate:   {learning_rate}
            Training size:   {n_train}
            Validation size: {n_val}
            Checkpoints:     {save_checkpoint}
            Device:          {device.type}
            Images scaling:  {img_scale}
            Mixed Precision: {amp}
        ''')
        
        params = {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': learning_rate,
            'training_set_size': n_train,
            'validation_set_size': n_val,
            'img_scale': img_scale,
            'mixed_precision': amp
        }
        
        mlflow.log_params(params)

        # Set up the optimizer, the loss, the learning rate scheduler and the loss scaling for AMP
        optimizer = optim.RMSprop(model.parameters(),
                                  lr=learning_rate, weight_decay=weight_decay, momentum=momentum, foreach=True)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5)  # goal: maximize Dice score
        grad_scaler = torch.cuda.amp.GradScaler(enabled=amp)
        criterion = nn.CrossEntropyLoss() if model.n_classes > 1 else nn.BCEWithLogitsLoss()
        global_step = 0

        # Begin training
        for epoch in range(1, epochs + 1):
            model.train()
            epoch_loss = 0
            with tqdm(total=n_train, desc=f'Epoch {epoch}/{epochs}', unit='img') as pbar:
                for batch in train_dataloader:
                    # images, msks_lv, msks_rv, msks_myo = batch['image'], batch['mask_lv'], batch['mask_rv'], batch['mask_myo']
                    images, masks = batch['image'], batch['masks']
                    images = images.permute(0, 3, 1, 2)
                    logging.info(f"images shape: {images.shape}")
    
                    assert images.shape[1] == model.n_channels, \
                        f'Network has been defined with {model.n_channels} input channels, ' \
                        f'but loaded images have {images.shape[1]} channels. Please check that ' \
                        'the images are loaded correctly.'
    
                    images = images.to(device=device, dtype=torch.float32) # , memory_format=torch.channels_last
                    true_masks = masks.to(device=device, dtype=torch.float32) # , memory_format=torch.channels_last
                    logging.info(f"true masks shape: {true_masks.shape}")
    
                    with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
                        # masks_pred = model(images).permute(1, 0, 2, 3)
                        masks_pred = model(images)
                        logging.info(f"masks pred shape: {masks_pred.shape}")
                        if model.n_classes == 1:
                            loss = criterion(masks_pred.squeeze(1), true_masks.float())
                            loss += dice_loss(F.sigmoid(masks_pred.squeeze(1)), true_masks.float(), multiclass=False)
                        else:
                            loss = criterion(masks_pred, true_masks)
                            loss += dice_loss(
                                F.sigmoid(masks_pred > 0.5).float(),
                                # F.softmax(masks_pred, dim=1).float(),
                                # F.one_hot(true_masks, model.n_classes).permute(0, 3, 1, 2).float(),
                                true_masks.float(),
                                multiclass=True
                            )
    
                    optimizer.zero_grad(set_to_none=True)
                    grad_scaler.scale(loss).backward()
                    grad_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
                    grad_scaler.step(optimizer)
                    grad_scaler.update()
    
                    pbar.update(images.shape[0])
                    global_step += 1
                    epoch_loss += loss.item()
                    pbar.set_postfix(**{'loss (batch)': loss.item()})
    
                    # Evaluation round
                    division_step = (n_train // (5 * batch_size))
                    if division_step > 0:
                        if global_step % division_step == 0:
                            val_score = evaluate(model, val_dataloader, device, amp)
                            scheduler.step(val_score)
                            logging.info('Validation Dice score: {}'.format(val_score))
    
            if save_checkpoint:
                Path(dir_checkpoint).mkdir(parents=True, exist_ok=True)
                state_dict = model.state_dict()
                # state_dict['mask_values'] = training_dataset.mask_values
                torch.save(state_dict, str(dir_checkpoint / 'checkpoint_epoch{}.pth'.format(epoch)))
                logging.info(f'Checkpoint {epoch} saved!')
        
        # Log train and validation set Dice Scores
        train_dice_score = evaluate(model, train_dataloader, device, amp)
        val_dice_score = evaluate(model, val_dataloader, device, amp)
        
        mlflow.log_metric("train_dice_score", train_dice_score)
        mlflow.log_metric("val_dice_score", val_dice_score)
        
        logging.info('Overall Train Dice score: {}'.format(train_dice_score))
        logging.info('Overall Validation Dice score: {}'.format(val_dice_score))

### Training

In [ ]:
# n_channels=3 for RGB images
# n_classes is the number of probabilities you want to get per pixel
model = FeedbackUNet(n_channels=1, n_classes=3, bilinear=False)
# model = FeedbackUNetV2(in_channels=1, classes=3)
# model = FeedbackUNetV3(n_channels=1, n_classes=3)
model = model.to(memory_format=torch.channels_last)

logging.info(f'Network:\n'
                 f'\t{model.n_channels} input channels\n'
                 f'\t{model.n_classes} output channels (classes)\n'
                 f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

model.to(device=device)

try:
    train_model(
        model=model,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        img_scale=scale,
        amp=amp
    )
except torch.cuda.OutOfMemoryError:
    logging.error('Detected OutOfMemoryError! '
                  'Enabling checkpointing to reduce memory usage, but this slows down training. '
                  'Consider enabling AMP (--amp) for fast and memory efficient training')
    torch.cuda.empty_cache()
    model.use_checkpointing()
    train_model(
        model=model,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        img_scale=scale,
        amp=amp
    )

### Testing and Evaluation

In [ ]:
def visualize_prediction(model, dataloader, device, sample_idx=0, img_idx=0):
    model.eval()
    with torch.no_grad():
        for i, sample in enumerate(dataloader):
            if i == sample_idx:
                # Extract data from the sample
                images = sample['image']
                true_masks = sample['masks']
                print(f'images shape: {images.shape}')
                print(f'true masks shape: {true_masks.shape}')
                images = images.permute(0, 3, 1, 2)
                images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                true_masks = true_masks.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)

                outputs = model(images)

                # Apply sigmoid to convert logits to probabilities, then threshold
                preds = torch.sigmoid(outputs)
                preds = (preds > 0.5).float()  # Convert to binary masks (0 or 1)
                preds = preds.cpu().numpy()
                print(f'preds shape: {preds.shape}')

                fig, ax = plt.subplots(2, 4, figsize=(15, 8))
                ax[0][0].imshow(images[img_idx].cpu().squeeze(), cmap='gray')
                ax[0][0].set_title('Input Image')

                ax[0][1].imshow(true_masks[img_idx][0].cpu().squeeze(), cmap='gray')
                ax[0][1].set_title('True LV')

                ax[0][2].imshow(true_masks[img_idx][1].cpu().squeeze(), cmap='gray')
                ax[0][2].set_title('True RV')

                ax[0][3].imshow(true_masks[img_idx][2].cpu().squeeze(), cmap='gray')
                ax[0][3].set_title('True MYO')

                ax[1][0].imshow(images[img_idx].cpu().squeeze(), cmap='gray')
                ax[1][0].set_title('Input Image')

                ax[1][1].imshow(preds[img_idx][0], cmap='gray')
                ax[1][1].set_title('Pred LV')

                ax[1][2].imshow(preds[img_idx][1], cmap='gray')
                ax[1][2].set_title('Pred RV')

                ax[1][3].imshow(preds[img_idx][2], cmap='gray')
                ax[1][3].set_title('Pred MYO')



                plt.show()
                break  # Visualize only one image
            else:
                continue

In [ ]:
visualize_prediction(model, test_dataloader, device, sample_idx=0, img_idx = 15)

In [ ]:
# torch.cuda.empty_cache()